# Recomendador basado en tópicos (LDA)

**Introducción al Procesamiento del Lenguaje Natural — TFI 1C 2026**

Esta es la otra estrategia de representación, distinta del recomendador con LLM. En vez de comparar palabras sueltas o delegar la decisión a un modelo generativo, modelo todo el corpus con **LDA** (modelado de tópicos, Unidad 3). La idea es que cada película queda descripta como una mezcla de tópicos latentes (por ejemplo "crimen/policial", "romance", "guerra"), y al usuario lo represento en ese mismo espacio de tópicos. Después recomiendo las películas más cercanas a su perfil.

Elegí LDA porque me da algo que el TF-IDF no: tópicos interpretables. Puedo mirar qué tópicos pesan en cada usuario y entender *por qué* recomienda lo que recomienda.

## 1. Datos

In [1]:
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from unidecode import unidecode
from nltk.stem import SnowballStemmer
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_csv("data/plots.csv").reset_index(drop=True)
users = pd.read_csv("data/perfiles_usuarios.csv")

## 2. Preprocesamiento

Mismo pipeline que vengo usando: saco acentos, puntuación y números, tiro stopwords (las de la materia más las de inglés porque las keywords vienen mezcladas) y aplico stemming. Junto sinopsis, keywords y género en un solo texto, repitiendo keywords y género para que pesen más.

In [2]:
sw_es = set(pd.read_csv("data/stop_words_spanish.csv")["word"].astype(str))
STOPWORDS = {unidecode(w).lower() for w in sw_es} | set(ENGLISH_STOP_WORDS)
stemmer = SnowballStemmer("spanish")

def preprocesar(texto):
    t = unidecode(str(texto)).lower()
    t = re.sub(r"[^a-z0-9\s]", " ", t)
    t = re.sub(r"\w*\d\w*", " ", t)
    tokens = [stemmer.stem(w) for w in t.split() if w not in STOPWORDS and len(w) > 2]
    return " ".join(tokens)

df["doc"] = (df["description"].map(preprocesar) + " " +
             (df["keywords"].map(preprocesar) + " ") * 2 +
             (df["genre"].map(preprocesar) + " ") * 2)

## 3. Modelo LDA

Para LDA uso `CountVectorizer` (frecuencias crudas, no TF-IDF: LDA es un modelo probabilístico sobre conteos de palabras). Entreno `LatentDirichletAllocation` con 20 tópicos, igual que se vio en la Unidad 3.

In [3]:
vectorizer = CountVectorizer(min_df=5, max_df=0.5)
X = vectorizer.fit_transform(df["doc"])
vocab = vectorizer.get_feature_names_out()

n_topicos = 20
lda = LatentDirichletAllocation(n_components=n_topicos, learning_method="batch",
                                random_state=1234)
theta = lda.fit_transform(X)        # distribución de tópicos por película
theta.shape

(4967, 20)

## 4. Qué capturan los tópicos

Antes de recomendar nada miro los tópicos, para chequear que tengan sentido. Cada tópico es una distribución sobre el vocabulario; muestro las palabras más probables de cada uno.

In [4]:
def palabras_topico(t, n=8):
    return [vocab[i] for i in lda.components_[t].argsort()[::-1][:n]]

for t in range(n_topicos):
    print(f"Tópico {t:2d}:", ", ".join(palabras_topico(t)))

Tópico  0: charact, music, biografi, titl, comedi, histori, quot, whit
Tópico  1: comedi, protagonist, timefram, femal, comedy, gay, romanc, famili
Tópico  2: terror, suspens, misteri, crim, kill, asesin, horror, violenci
Tópico  3: comedi, romanc, viaj, edad, carreter, pelicul, amor, voyeur
Tópico  4: terror, aventur, vampir, misteri, bas, accion, kill, detectiv
Tópico  5: documental, referenc, franci, misteri, histori, sigl, bas, gam
Tópico  6: relacion, romanc, sex, hij, comedi, sexual, herman, padr
Tópico  7: crim, accion, polici, agent, suspens, asesinat, investig, fbi
Tópico  8: comedi, comic, book, accion, reality, arizon, superher, puebl
Tópico  9: misteri, terror, suspens, dead, muert, body, submarin, accident
Tópico 10: anim, aventur, film, fantasi, animal, accion, comedi, cult
Tópico 11: guerr, movi, belic, histori, accion, mundial, vietnam, war
Tópico 12: femal, nudity, frontal, mal, rear, desnudez, bar, hair
Tópico 13: cienci, ficcion, accion, aventur, alienigen, futur, vi

## 5. El usuario en el espacio de tópicos

Represento al usuario combinando dos distribuciones de tópicos: la del **historial** (promedio de los tópicos de las películas que vio) y la de la **query** (la proyecto al modelo con `lda.transform`). Las combino con un peso `alpha`. Si una película del historial no está en el corpus la salteo (trabajo con el df crudo).

In [5]:
name_to_idx = {n: i for i, n in enumerate(df["name"])}
hist_cols = [c for c in users.columns if c.startswith("pelicula_")]

def indices_historial(row):
    return [name_to_idx[row[c]] for c in hist_cols
            if pd.notna(row[c]) and row[c] in name_to_idx]

def perfil_topicos(row, alpha=0.5):
    idxs = indices_historial(row)
    t_hist = theta[idxs].mean(axis=0) if idxs else None
    q_bow = vectorizer.transform([preprocesar(row["query"])])
    t_query = lda.transform(q_bow)[0] if q_bow.nnz else None
    if t_hist is not None and t_query is not None:
        v = alpha * t_hist + (1 - alpha) * t_query
    else:
        v = t_hist if t_hist is not None else t_query
    return v, idxs

def recomendar(row, top_n=5, alpha=0.5):
    v, idxs = perfil_topicos(row, alpha)
    sims = cosine_similarity(v.reshape(1, -1), theta).flatten()
    excluir = set(idxs)
    recs, vistos = [], set()
    for i in np.argsort(-sims):
        if i in excluir or df.iloc[i]["name"] in vistos:
            continue
        vistos.add(df.iloc[i]["name"])
        recs.append((df.iloc[i]["name"], df.iloc[i]["genre"], float(sims[i])))
        if len(recs) == top_n:
            break
    return recs

## 6. Resultados

In [6]:
for _, r in users.iterrows():
    print(f'\n{r["nombre"]} ({r["tipo_perfil"]}) — "{r["query"]}"')
    for nombre, genero, score in recomendar(r):
        print(f'  - {nombre[:45]:45} [{genero[:24]}]  {score:.2f}')


Valentina (definido) — "Quiero una película donde una mujer enfrenta una amenaza invisible que viene de alguien cercano"
  - Hardware: Programado para matar               [ciencia ficción, suspens]  0.80
  - El engendro del diablo                        [terror]  0.79
  - Jeepers Creepers 2                            [terror]  0.77
  - La posesión                                   [drama, terror]  0.77
  - Blade: Trinity                                [acción, terror, ciencia ]  0.75

Rodrigo (definido) — "Busco algo basado en hechos reales sobre corrupción o poder político"
  - Starsky &amp; Hutch                           [comedia, crimen]  0.92
  - Punisher 2: Zona de guerra                    [acción, crimen, drama]  0.92
  - Canción triste de Hill Street                 [crimen, drama, misterio]  0.91
  - Bajo el fuego                                 [drama, bélico]  0.89
  - The Staircase                                 [documental, crimen, dram]  0.88

Camila (definido) — "Una 

Para los dos casos de contraste muestro además los tópicos dominantes del perfil del usuario, que es lo que hace interpretable a esta estrategia.

In [7]:
def explicar(nombre):
    r = users[users["nombre"] == nombre].iloc[0]
    v, _ = perfil_topicos(r)
    print(f'{r["nombre"]} ({r["tipo_perfil"]}) — "{r["query"]}"')
    print("Tópicos dominantes del perfil:")
    for t in v.argsort()[::-1][:3]:
        print(f'  T{t} ({v[t]:.2f}): {", ".join(palabras_topico(t, 6))}')
    print("Recomendaciones:")
    for nombre_, genero, score in recomendar(r):
        print(f'  - {nombre_}  [{genero}]')

explicar("Rodrigo")
print("-" * 70)
explicar("Paula")

Rodrigo (definido) — "Busco algo basado en hechos reales sobre corrupción o poder político"
Tópicos dominantes del perfil:
  T7 (0.28): crim, accion, polici, agent, suspens, asesinat
  T0 (0.23): charact, music, biografi, titl, comedi, histori
  T11 (0.18): guerr, movi, belic, histori, accion, mundial
Recomendaciones:
  - Starsky &amp; Hutch  [comedia, crimen]
  - Punisher 2: Zona de guerra  [acción, crimen, drama]
  - Canción triste de Hill Street  [crimen, drama, misterio]
  - Bajo el fuego  [drama, bélico]
  - The Staircase  [documental, crimen, drama]
----------------------------------------------------------------------
Paula (ambiguo) — "No tengo ganas de pensar mucho, pero tampoco quiero algo vacío, algo intermedio"
Tópicos dominantes del perfil:
  T17 (0.28): romanc, comedi, mal, bar, navid, relacion
  T5 (0.28): documental, referenc, franci, misteri, histori, sigl
  T6 (0.13): relacion, romanc, sex, hij, comedi, sexual
Recomendaciones:
  - Mi obsesión por Helena  [drama, miste

## 7. Evaluación

Sin ground truth uso la misma proxy que en la otra estrategia: cuánto se solapan los géneros de las recomendaciones con los del historial. Espero más coherencia en los perfiles definidos que en los ambiguos.

In [8]:
def generos(idxs):
    g = set()
    for i in idxs:
        g.update(x.strip() for x in str(df.iloc[i]["genre"]).split(","))
    return g

filas = []
for _, r in users.iterrows():
    g_hist = generos(indices_historial(r))
    idxs_rec = [name_to_idx[n] for n, _, _ in recomendar(r) if n in name_to_idx]
    g_rec = generos(idxs_rec)
    solap = len(g_hist & g_rec) / len(g_rec) if g_rec else 0.0
    filas.append((r["nombre"], r["tipo_perfil"], round(solap, 2)))

ev = pd.DataFrame(filas, columns=["nombre", "tipo_perfil", "solapamiento_genero"])
print(ev.to_string(index=False))
print("\nPromedio por tipo:")
print(ev.groupby("tipo_perfil")["solapamiento_genero"].mean().round(2))

   nombre tipo_perfil  solapamiento_genero
Valentina    definido                 0.80
  Rodrigo    definido                 0.43
   Camila    definido                 1.00
    Tomás    definido                 0.62
    Lucía    definido                 1.00
   Martín    definido                 1.00
    Sofía    definido                 0.80
    Diego    definido                 1.00
    Elena    definido                 0.50
  Facundo    definido                 0.50
   Julián     ambiguo                 0.50
  Mariana     ambiguo                 0.17
  Nicolás     ambiguo                 0.50
    Paula     ambiguo                 0.50

Promedio por tipo:
tipo_perfil
ambiguo     0.42
definido    0.76
Name: solapamiento_genero, dtype: float64


## 8. Qué gana y qué pierde esta representación

LDA me da tópicos interpretables y un perfil de usuario que puedo leer ("este usuario carga en el tópico de crimen/policial"). Eso es lo que la diferencia del TF-IDF y la hace defendible.

El costo es que, con sinopsis cortas y ruidosas (y mezcla de idiomas en las keywords), los tópicos quedan algo difusos y de grano grueso: captura bien el tema general pero no los matices finos de una query ("algo que te haga pensar sobre qué es real"). También depende del número de tópicos elegido. Frente al LLM, LDA no entiende intención ni negaciones, pero a cambio es transparente, determinista y rápido.